### **1\. Prouducts Not Being Sold**

**Proposition:** Get a list of Products that are not making sales but are not discontinued, to determine why they have fallen out of flavor

**Tables:** Production.Product, Sales.SalesOrderDetail

In [ ]:
-- 1
-- Proposition: Get a list of Products that are not making sales but are not discontinued, to determine why they have fallen out of flavor

USE AdventureWorks2022

SELECT P.ProductID, P.Name
FROM Production.Product as P
WHERE P.ProductID NOT IN (SELECT ProductID FROM Sales.SalesOrderDetail)

INTERSECT

SELECT P.ProductID, P.Name
FROM Production.Product as P
WHERE P.DiscontinuedDate IS NULL
ORDER BY ProductID


### **2\. Sales People with Low Sales**

**Proposition:** Get a list of sales people with low sales and the stores they are responsible for. This can be used to determine if the sales person needs to be swapped or if there is an issue with the store.

**Tables:** Person.BusinessEntity, Sales.SalesPerson, Sales.Store

In [ ]:
-- 2
-- Proposition: Get a list of sales people with low sales and the stores they are responsible for. This can be used to determine if the sales person needs to be swapped or if there is an issue with the store.

USE AdventureWorks2022;

WITH LowSales AS (
    SELECT BE.BusinessEntityID 
    FROM Person.BusinessEntity as BE

    INTERSECT

    SELECT BusinessEntityID
    FROM Sales.SalesPerson as SP
    WHERE SP.SalesYTD < SP.SalesLastYear
)

SELECT S.Name, S.SalesPersonID
FROM LowSales as LS
    INNER JOIN Sales.Store as S
        ON LS.BusinessEntityID = S.SalesPersonID

### **3\. Products with Special Orders**

**Proposition:** Get a list of products with special orders. This can be used to provide a catelogue of that info.

**Tables:** Sales.SpecialOfferProduct, Production.Product

In [ ]:
-- 3
-- Proposition: Get a list of products with special orders. This can be used to provide a catelogue of that info.
USE AdventureWorks2022;

WITH ProductSpecialOffer AS (
    SELECT SOP.ProductID
    FROM Sales.SpecialOfferProduct as SOP

    INTERSECT

    SELECT P.ProductID
    FROM Production.Product as P
)

SELECT PP.ProductID, PP.Name, PP.ListPrice
FROM ProductSpecialOffer as PSO
    INNER JOIN Production.Product as PP
        ON PSO.ProductID = PP.ProductID

### **4\. List of Products & Their Pictures**

**Proposition:** Get a list of products with their pictures. This can be used to provide a catelogue of that info.

**Tables:** Sales.SpecialOfferProduct, Production.Product, Production.ProductProductPhoto, Production.ProductPhoto

In [ ]:
-- 4
-- Proposition: Get a list of products with their pictures. This can be used to provide a catelogue of that info.

USE AdventureWorks2022;

WITH ProductSpecialOffer AS (
    SELECT SOP.ProductID
    FROM Sales.SpecialOfferProduct as SOP

    INTERSECT

    SELECT P.ProductID
    FROM Production.Product as P
),
SpecialProduct AS (
    SELECT PP.ProductID, PP.Name, PP.ListPrice
        FROM ProductSpecialOffer as PSO
            INNER JOIN Production.Product as PP
                ON PSO.ProductID = PP.ProductID
)

SELECT SP.ProductID, SP.ListPrice, PP.*
FROM SpecialProduct as SP
    INNER JOIN Production.ProductProductPhoto as PPP
        ON SP.ProductID = PPP.ProductID
    INNER JOIN Production.ProductPhoto as PP
        ON PPP.ProductPhotoID = PP.ProductPhotoID

### **5\. Stores & Their Shipping Locations**

**Proposition:** Get a list of stores and their shipping locations to be able to determine the proper way to drop off the product.

**Tables:** Person.BusinessEntityAddress, Sales.Store, Person.Address

In [ ]:
-- 5
-- Proposition: Get a list of stores and their shipping locations to be able to determine the proper way to drop off the product.

USE AdventureWorks2022;

WITH ShippingLocations AS (
    SELECT BEA.BusinessEntityID
    FROM Person.BusinessEntityAddress as BEA
    WHERE BEA.AddressTypeID = 5

    INTERSECT

    SELECT S.BusinessEntityID
    FROM Sales.Store as S
)
 SELECT SL.BusinessEntityID, A.AddressLine1, A.AddressLine2, A.PostalCode
 FROM ShippingLocations as SL
    INNER JOIN Person.BusinessEntityAddress as BEA
        ON SL.BusinessEntityID = BEA.BusinessEntityID
    INNER JOIN Person.Address as A
        ON BEA.AddressID = A.AddressID

### **6\. Addresses of CEOs and Sales Managers**

**Proposition:** Get a list of CEO and Sals Manager addresses, to send a gift for business

**Tables:** Person.BusinessEntityContact, Person.ContactType, Person.BusinessEntityAddress, Person.Address

In [ ]:
-- 6
-- Proposition: Get a list of CEO and Sals Manager addresses, to send a gift for business

USE AdventureWorks2022;

WITH CEOSalesManager AS (
    SELECT BEC.BusinessEntityID, CT.Name as ContactType
    FROM Person.BusinessEntityContact as BEC
        INNER JOIN Person.ContactType as CT
            ON BEC.ContactTypeID = CT.ContactTypeID
    WHERE CT.Name = 'Owner'

    UNION

    SELECT BEC.BusinessEntityID, CT.Name as ContactType
    FROM Person.BusinessEntityContact as BEC
        INNER JOIN Person.ContactType as CT
            ON BEC.ContactTypeID = CT.ContactTypeID
    WHERE CT.Name = 'Sales Manager'
)

SELECT CSM.*, A.*
FROM CEOSalesManager as CSM
    INNER JOIN Person.BusinessEntityAddress as BEA
        ON CSM.BusinessEntityID = BEA.BusinessEntityID
    INNER JOIN Person.Address as A
        ON BEA.AddressID = A.AddressID

### **7\. US Cities & States Work Is Conducted**

**Proposition:** Get a list of US Cities and States where business is done in order to determine new places of expansion.

**Tables:** Person.BusinessEntityAddress, Sales.Store, Person.Address, Person.StateProvince

In [ ]:
-- 7
-- Proposition: Get a list of US Cities and States where business is done in order to determine new places of expansion.

USE AdventureWorks2022;

WITH Stores AS (
    SELECT BEA.BusinessEntityID
    FROM Person.BusinessEntityAddress as BEA

    INTERSECT

    SELECT S.BusinessEntityID
    FROM Sales.Store as S
),
StoreAddress AS (
    SELECT BEA.*
    FROM Stores as S
        INNER JOIN Person.BusinessEntityAddress as BEA
            ON BEA.BusinessEntityID = S.BusinessEntityID
),
StoreAddresses AS (
    SELECT SA.BusinessEntityID, SA.AddressID, SA.AddressTypeID, A.AddressLine1, 
        A.AddressLine2, A.City, A.PostalCode, A.StateProvinceID
    FROM StoreAddress as SA
        INNER JOIN Person.Address as A
            ON SA.AddressID = A.AddressID
)

SELECT SP.Name, SAS.City
FROM StoreAddresses AS SAS
    INNER JOIN Person.StateProvince AS SP
        ON SAS.StateProvinceID = SP.StateProvinceID
WHERE SP.CountryRegionCode = 'US'

GROUP BY SP.Name, SAS.City
ORDER BY SP.Name, SAS.City

### **8\. Countries, States & Provinces Work Is Conducted**

**Proposition:** Get a list of Countries, States & Provinces where business is done in order to determine new places of expansion.

**Tables:** Person.BusinessEntityAddress, Sales.Store, Person.Address, Person.StateProvince, Person.CountryRegion

In [ ]:
-- 8
-- Proposition: Get a list of Countries, States & Provinces where business is done in order to determine new places of expansion.

USE AdventureWorks2022;

WITH Stores AS (
    SELECT BEA.BusinessEntityID
    FROM Person.BusinessEntityAddress as BEA

    INTERSECT

    SELECT S.BusinessEntityID
    FROM Sales.Store as S
),
StoreAddress AS (
    SELECT BEA.*
    FROM Stores as S
        INNER JOIN Person.BusinessEntityAddress as BEA
            ON BEA.BusinessEntityID = S.BusinessEntityID
),
StoreAddresses AS (
    SELECT SA.BusinessEntityID, SA.AddressID, SA.AddressTypeID, A.AddressLine1, 
        A.AddressLine2, A.City, A.PostalCode, A.StateProvinceID
    FROM StoreAddress as SA
        INNER JOIN Person.Address as A
            ON SA.AddressID = A.AddressID
)

SELECT CR.Name, SP.Name
FROM StoreAddresses AS SAS
    INNER JOIN Person.StateProvince AS SP
        ON SAS.StateProvinceID = SP.StateProvinceID
    INNER JOIN Person.CountryRegion AS CR
        ON SP.CountryRegionCode = CR.CountryRegionCode
GROUP BY CR.Name, SP.Name
ORDER BY CR.Name, SP.Name


### **9\. Employees with a lot of Vaction and Sick Hours**

**Proposition:** Get a list of employees with a lot of sick and vacation hours to determine if anyone is abusing it.

**Tables:** HumanResources.Employee

In [ ]:
-- 9
-- Proposition: Get a list of employees with a lot of sick and vacation hours to determine if anyone is abusing it.

USE AdventureWorks2022;

WITH Hooky AS (
    SELECT *
    FROM HumanResources.Employee as E


    EXCEPT

    SELECT *
    FROM HumanResources.Employee as E
    WHERE E.SickLeaveHours < 50 AND E.VacationHours < 50
)

SELECT Hooky.BusinessEntityID, Hooky.JobTitle, Hooky.SickLeaveHours, Hooky.VacationHours, Hooky.SalariedFlag
FROM Hooky
WHERE Hooky.JobTitle != 'Chief Executive Officer'
ORDER BY JobTitle

<span style="color: #800000;font-weight: bold;">### </span> <span style="color: #000080;font-weight: bold;"><strong>10. Employee Pay Roll</strong></span>
<span style="color: #000080;font-weight: bold;"><strong>Proposition:</strong></span> Get a list of employees and their payment history to start pay roll.
<span style="color: #000080;font-weight: bold;"><strong>Tables:</strong></span> HumanResources.Employee, <span style="color: rgb(33, 33, 33); font-family: Menlo, Monaco, &quot;Courier New&quot;, monospace; font-size: 12px; white-space: pre;">HumanResources.EmployeePayHistory</span>

In [192]:
-- 10
-- Proposition: Get a list of employees and their payment history to start pay roll.

USE AdventureWorks2022;

WITH PayRoll AS (
    SELECT *
    FROM HumanResources.Employee as E


    INTERSECT

    SELECT *
    FROM HumanResources.Employee as E
    WHERE E.JobTitle NOT LIKE 'Chief%'
)
SELECT BelowExecutive.BusinessEntityID, BelowExecutive.JobTitle, BelowExecutive.SalariedFlag, 
    EPH.Rate, EPH.PayFrequency
FROM(
    SELECT *
    FROM PayRoll as PR

    INTERSECT

    SELECT *
        FROM HumanResources.Employee as E
        WHERE E.JobTitle NOT LIKE 'Vice President%'
) as BelowExecutive
    INNER JOIN HumanResources.EmployeePayHistory as EPH
        ON BelowExecutive.BusinessEntityID = EPH.BusinessEntityID